# 08 — Deep Learning Projects

This notebook demonstrates an end-to-end deep-learning project workflow with a synthetic tabular binary-classification dataset. It covers data splitting, model training, evaluation, and saving/reloading a Keras model. The dataset is generated locally.

## 1. Setup

Install a TensorFlow version supported by your Python/platform using the [official guide](https://www.tensorflow.org/install).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow version:', tf.__version__)

## 2. Define the task and generate synthetic data

Task: classify synthetic records into one of two labels using numeric features. This makes the notebook self-contained; it is a workflow exercise rather than a benchmark.

In [ ]:
rng = np.random.default_rng(SEED)
n_samples, n_features = 1800, 10
X_raw = rng.normal(size=(n_samples, n_features)).astype('float32')
latent = (1.3 * X_raw[:, 0] - 1.0 * X_raw[:, 1] + 0.7 * X_raw[:, 2] * X_raw[:, 3]
          + 0.45 * rng.normal(size=n_samples))
y = (latent > np.median(latent)).astype('float32')
print('Features:', X_raw.shape, 'Labels:', y.shape)
print('Class balance:', np.bincount(y.astype(int)))

## 3. Split before fitting preprocessing

Use separate training, validation, and test sets. Fit the scaler on training data only to avoid leaking information from validation or test data.

In [ ]:
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype('float32')
X_val = scaler.transform(X_val_raw).astype('float32')
X_test = scaler.transform(X_test_raw).astype('float32')
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)

## 4. Establish a simple baseline

A majority-class baseline is a minimal reference. For a real tabular task, also compare a suitable conventional model such as logistic regression or gradient-boosted trees.

In [ ]:
majority_class = int(np.bincount(y_train.astype(int)).argmax())
baseline_predictions = np.full(y_test.shape, majority_class, dtype=int)
print('Majority-class baseline accuracy:', round(accuracy_score(y_test, baseline_predictions), 4))

## 5. Build and train a Keras model

Early stopping monitors validation loss and restores the best weights. The test set is not used for training or callback decisions.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
print('Epochs run:', len(history.history['loss']))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy')
plt.title('Training and validation loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Evaluate held-out test data

Use metrics that match the problem. Accuracy alone can be misleading when classes are imbalanced or errors have different costs.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print(classification_report(y_test.astype(int), predictions, target_names=['class_0', 'class_1'], zero_division=0))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test.astype(int), predictions, display_labels=['class_0', 'class_1'], ax=ax)
ax.set_title('Test confusion matrix')
plt.tight_layout()
plt.show()

## 7. Inspect prediction examples

For real projects, inspect correct and incorrect predictions with relevant context. The synthetic labels are generated by a known formula and should not be interpreted as a real application.

In [ ]:
for i in range(min(10, len(X_test))):
    print(f'true={int(y_test[i])}, predicted={predictions[i]}, probability={probabilities[i]:.3f}')
errors = np.flatnonzero(predictions != y_test.astype(int))
print('Number of test errors:', len(errors))
print('First error indices:', errors[:10])

## 8. Save, reload, and verify predictions

The `.keras` file is written to the notebook's current working directory.

In [ ]:
model_path = 'deep_learning_project.keras'
model.save(model_path)
loaded_model = keras.models.load_model(model_path)
before = model.predict(X_test[:20], verbose=0)
after = loaded_model.predict(X_test[:20], verbose=0)
max_diff = float(np.max(np.abs(before - after)))
print('Saved model:', model_path)
print('Maximum prediction difference:', max_diff)
assert np.allclose(before, after, atol=1e-6)

## 9. Project report template

Document: (1) task and intended use, (2) dataset provenance and license, (3) split strategy and preprocessing, (4) baseline, (5) architecture and training settings, (6) test metrics, (7) error analysis, (8) saved-model inference, (9) limitations and risks, and (10) next experiments.

## Key takeaways

- Define the task and metric before training.
- Compare against a baseline.
- Fit preprocessing on training data only.
- Use validation data for model selection and test data for final evaluation.
- Review errors, document limitations, and save reproducible artifacts.

Next: complete `practice.md`, then review `solution.md`.